In [2]:
import re
import nltk
from collections import defaultdict, Counter
from datetime import datetime

# Download required NLTK resources (run once)
nltk.download('punkt')
nltk.download('stopwords')

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

# Sample Clinical Text Input

clinical_text = """
Patient was admitted on 12 January 2023 with complaints of fever and cough.
On 15 January 2023, diagnosed with pneumonia.
Antibiotics were started on 16 January 2023.
On 20 February 2023, follow-up showed improvement.
On 10 March 2023, patient discharged.
"""


# Date Extraction

def extract_dates(sentence):
    date_pattern = r'\d{1,2} \w+ \d{4}'
    matches = re.findall(date_pattern, sentence)
    return matches

# Simple Medical Keyword Detection

medical_keywords = [
    "diagnosed", "admitted", "discharged",
    "antibiotics", "fever", "cough",
    "pneumonia", "follow-up"
]

def detect_medical_events(sentence):
    events = []
    for word in medical_keywords:
        if word.lower() in sentence.lower():
            events.append(word)
    return events

# Build Timeline

def build_timeline(text):
    timeline = []
    sentences = sent_tokenize(text)
    
    for sentence in sentences:
        dates = extract_dates(sentence)
        events = detect_medical_events(sentence)
        
        if dates:
            for date in dates:
                try:
                    parsed_date = datetime.strptime(date, "%d %B %Y")
                    timeline.append((parsed_date, sentence.strip()))
                except:
                    pass

    timeline.sort(key=lambda x: x[0])
    return timeline


# Extractive Summarisation

def summarize_text(text, num_sentences=3):
    stop_words = set(stopwords.words('english'))
    sentences = sent_tokenize(text)
    
    word_frequencies = Counter()
    for word in word_tokenize(text.lower()):
        if word.isalnum() and word not in stop_words:
            word_frequencies[word] += 1
    
    sentence_scores = {}
    for sentence in sentences:
        for word in word_tokenize(sentence.lower()):
            if word in word_frequencies:
                sentence_scores[sentence] = sentence_scores.get(sentence, 0) + word_frequencies[word]
    
    ranked_sentences = sorted(sentence_scores, key=sentence_scores.get, reverse=True)
    return ranked_sentences[:num_sentences]

# Execute Pipeline

timeline = build_timeline(clinical_text)
summary = summarize_text(clinical_text)

# Output Results

print("\n📅 Patient Timeline:\n")
for date, event in timeline:
    print(date.strftime("%d-%m-%Y"), ":", event)

print("\n📝 Generated Summary:\n")
for sent in summary:
    print("-", sent)


📅 Patient Timeline:

12-01-2023 : Patient was admitted on 12 January 2023 with complaints of fever and cough.
15-01-2023 : On 15 January 2023, diagnosed with pneumonia.
16-01-2023 : Antibiotics were started on 16 January 2023.
20-02-2023 : On 20 February 2023, follow-up showed improvement.
10-03-2023 : On 10 March 2023, patient discharged.

📝 Generated Summary:

- 
Patient was admitted on 12 January 2023 with complaints of fever and cough.
- On 15 January 2023, diagnosed with pneumonia.
- Antibiotics were started on 16 January 2023.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Shaarukesh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Shaarukesh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
